In [66]:
import goEmotions
import re
from emoji import demojize
import time
import pickle
import random
import os
from datetime import datetime
import pytz
from dotenv import load_dotenv
import sqlite3
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException

load_dotenv();

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/104.0.5112.79 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Version/15.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/104.0.5112.79 Safari/537.36"
]

/home/andrew/.local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model successfully loaded!


In [2]:
def setupDriver():
    # Set up Brave options
    options = Options()
    options.binary_location = "/usr/bin/brave-browser"  # Update with your Brave installation path

    # Choose a random User-Agent from the list
    user_agent = random.choice(USER_AGENTS)
    options.add_argument(f"user-agent={user_agent}")
    options.add_argument("--disable-blink-features=AutomationControlled")  # Helps evade bot detection

    # Define the user data directory where cookies will be stored
    user_data_dir = os.path.join(os.getcwd(), "cookies")  # Create a 'cookies' directory in the current working directory
    options.add_argument(f"user-data-dir={user_data_dir}")  # Use this directory for storing cookies

    # Create a WebDriver instance using the Brave browser (ensure ChromeDriver is in PATH)
    driver = webdriver.Chrome(options=options)
    return driver

In [3]:
# Function to check if the account menu exists, indicating a successful login
def checkAccountLoggedIn(driver):
    if driver.current_url == "https://x.com/home":
        print("Check Account Logged in = true")
        return True
    print("Check Account Logged in = false")
    return False

In [4]:
# Function to perform login (if needed) and handle cookies
def login():
    driver = setupDriver()

    # Try to load cookies from previous session if they exist
    try:
        driver.get("https://x.com")  # Navigate to the homepage or login page to check if we have cookies
        time.sleep(10)  # Wait for the page to load
        
        # Check if the account menu is available, meaning we're already logged in
        if checkAccountLoggedIn(driver):
            print("Logged in using cookies.")
            return driver  # Return the driver with cookies applied
        else:
            print("Cookies did not work. Logging in manually...")
            raise Exception("Cookies invalid or expired, logging in manually.")  # Force manual login

    except Exception as e:
        print(f"Error: {e}")
        print("Logging in manually...")
        driver.get("https://x.com/i/flow/login")
        time.sleep(10)  # Wait for login page to load

        # Manually log in (provide your credentials here)
        username_field = driver.find_element(By.NAME, "text")
        username_field.send_keys(os.getenv("twitterEmail"))
        driver.find_element(By.XPATH, "//span[text()='Next']").click()
        time.sleep(6)

        password_field = driver.find_element(By.NAME, "password")
        password_field.send_keys(os.getenv("twitterPassword"))
        driver.find_element(By.XPATH, "//span[text()='Log in']").click()
        time.sleep(7)

        print("Successfully Logged in")
        return driver  # Return the logged-in driver

In [58]:
def preprocessingTweet(text):
    text = demojize(text); # Convert emoji to text
    text = re.sub(r'@elonmusk', 'Elon Musk', text) # Replace @elonmusk to "Elon Musk"
    text = re.sub(r'@\w+', '[MENTION]', text) # Remove @ to another user
    text = re.sub("\n+", " ", text) # Remove \n
    text = text.strip() # Remove extra spaces
    return text

In [59]:
def convertToCentral(utc_time_str):
    # Define the UTC timezone
    utc_zone = pytz.utc
    # Define the Central timezone (CST/CDT)
    central_zone = pytz.timezone('US/Central')

    # Parse the input UTC time string into a datetime object
    utc_time = datetime.strptime(utc_time_str, "%Y-%m-%dT%H:%M:%S.%fZ")
    utc_time = utc_zone.localize(utc_time)  # Localize to UTC

    # Convert to Central Time
    central_time = utc_time.astimezone(central_zone)

    # Return the Central time in string format
    return central_time.strftime('%Y-%m-%d %I:%M:%S %p')  # 12-hour format

In [69]:
def extractTweets(driver):
    tweetsAdded = 0 # Count how many unique tweets added to the database
    conn = sqlite3.connect("rtsProjectDB.db")
    cursor = conn.cursor()
    
    try:
        # Extract tweet containers (articles with data-testid="tweet")
        tweetElements = driver.find_elements(By.XPATH, "//article[@data-testid='tweet']")
        # print(f"Elements found {len(tweetElements)}")
        
        for tweet in tweetElements:
            try:
                # Get the div with tweetText (multiple spans with tweet content)
                tweetTextElement = tweet.find_element(By.XPATH, ".//div[@data-testid='tweetText']")
                
                # Get tweet ID
                tweet_link = tweet.find_element(By.XPATH, ".//a[contains(@href, '/status/')]").get_attribute("href")
                matchID = re.search(r'/status/(\d+)', tweet_link)
                tweet_id = matchID.group(1) # Get digit portion

                # Check if tweet is already in the database
                cursor.execute('SELECT 1 FROM tweets WHERE id = ?', (tweet_id,))
                existing_tweet = cursor.fetchone()
                if existing_tweet: # Skip rest of tweets if a duplicate is found
                    print(f"Duplicate tweet found. Skipping the rest...")
                    break
                
                # Get tweet text (including emoji)
                tweet_text = ""
                # Iterate over all child elements (spans and img) in the tweetText div
                tweet_children = tweetTextElement.find_elements(By.XPATH, ".//span | .//img")
                for child in tweet_children:
                    if child.tag_name == 'span':
                        tweet_text += child.text # If it's a span, get the text
                    elif child.tag_name == 'img':
                        tweet_text += child.get_attribute("alt") # If img, get emoji in the alt attribute
                
                # Get tweet metadata (reply count, retweet count, like count, view count)
                reply_count = tweet.find_element(By.XPATH, ".//button[@data-testid='reply']//span").text
                retweet_count = tweet.find_element(By.XPATH, ".//button[@data-testid='retweet']//span").text
                like_count = tweet.find_element(By.XPATH, ".//button[@data-testid='like']//span").text
                try: # If no views, there won't be a span
                    view_count = tweet.find_element(By.XPATH, ".//a[contains(@aria-label, 'views')]//span").text
                except:
                    view_count = 0
                # Default to 0 if no text is found for any of the counts
                reply_count = reply_count if reply_count else "0"
                retweet_count = retweet_count if retweet_count else "0"
                like_count = like_count if like_count else "0"
                # view_count = view_count if view_count else "0"

                # Get tweet creation date (timestamp)
                created_date = tweet.find_element(By.XPATH, ".//time").get_attribute("datetime")

                
                # Store tweet data in a dictionary
                tweetData = [
                    tweet_id,
                    tweet_text,
                    preprocessingTweet(tweet_text),
                    reply_count,
                    view_count,
                    like_count,
                    retweet_count,
                    convertToCentral(created_date)
                ]
                # Get sentiment scores from the clean text
                sentimentScores = goEmotions.getTextSentiment(tweetData[2])
                for category in sentimentScores:
                    tweetData.append(sentimentScores[category])

                # Add to database
                cursor.execute('''
                    INSERT INTO tweets (id, origText, cleanText, replyCount, viewCount, likeCount, retweetCount, createdDate,
                            Positive, Hopeful, Pride, Approval, Curiosity, Fear, Remorse, Sadness, Disapproval, Neutral)
                    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                ''', tweetData)
                conn.commit()

                tweetsAdded += 1

                if (tweetsAdded > 5):
                    break;
                
            except Exception as e:
                print(f"Error extracting data from tweet: {e}")
        
    except Exception as e:
        print(f"Error while extracting tweets: {e}")
    
    conn.close()
    print(f"{tweetsAdded} tweets added!")


In [6]:
def tempLogin():
    driver = setupDriver()
    #driver.get("https://x.com/i/flow/login")

    driver.get("https://x.com")  # Navigate to the homepage or login page to check if we have cookies
    time.sleep(5)  # Wait for the page to load
        
    # Check if the account menu is available, meaning we're already logged in
    if checkAccountLoggedIn(driver):
        print("Logged in using cookies.")
        return driver  # Return the driver with cookies applied

In [7]:
driver = tempLogin()

Check Account Logged in = false


In [11]:
driver = login()
# driver = setupDriver()
# driver.get("https://x.com")

Check Account Logged in = true
Logged in using cookies.


In [ ]:
driver.get("https://x.com/search?q=%24TSLA%20lang%3Aen%20-filter%3Alinks&f=live&src=typed_query")

In [70]:
extractTweets(driver)

6 tweets added!


In [71]:
driver.quit()